# NB7 - Train Model 2: IncongruityClassifier (Person D)
**Run this AFTER NB6 is complete and `mentalmanip-features` is shared as a Kaggle dataset.**

## Setup:
1. `+ Add Data` -> search `mentalmanip-features` -> add it
2. GPU optional (small model -> CPU is fine)
3. Run all cells

Trains a 3-class classifier: Genuine / Sarcasm / Manipulation
Output: `incongruity_classifier.pt` -> share as Kaggle dataset `incongruity-classifier`
Person C uses it in NB8 (pipeline).


## 0. Install

In [ ]:
%%capture
!pip install scikit-learn scipy matplotlib seaborn xgboost -q
print('done')

## 1. Config

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from xgboost import XGBClassifier

FEATURES_CSV = '/kaggle/input/mentalmanip-features/mentalmanip_features.csv'
OUT_DIR      = '/kaggle/working'
CLASS_NAMES  = ['Genuine', 'Sarcasm', 'Manipulation']
N_CLASSES    = 3
N_FEATURES   = 46   # 42 emotion/context features + 4 lexical features
BATCH_SIZE   = 64
EPOCHS       = 50
PATIENCE     = 15
LR           = 1e-3
SEED         = 42
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_S1_XGB = os.path.join(OUT_DIR, 'stage1_genuine_vs_rest.json')   # XGBoost (JSON)
CKPT_S2      = os.path.join(OUT_DIR, 'stage2_sarcasm_vs_manip.pt')
CKPT_FINAL   = os.path.join(OUT_DIR, 'incongruity_classifier.pt')

torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')
print('Two-stage: Stage1=XGBoost (Genuine vs Rest) | Stage2=MLP (Sarcasm vs Manipulation)')

## 2. Load & Split Data
Split by `dialogue_id` to prevent turn-level leakage across train/val/test.

In [ ]:
df = pd.read_csv(FEATURES_CSV)
print(f'Loaded: {df.shape}  |  Raw label dist: {dict(df["label"].value_counts().sort_index())}')

# Remap label 3 (Ambiguous) → 2 (Manipulation) — borderline manipulation cases
# with low annotator agreement are indistinguishable in 42-dim feature space.
df['label'] = df['label'].replace(3, 2)
print(f'Label dist after remap: {dict(df["label"].value_counts().sort_index())}')

NON_FEATURE_COLS = {
    'dialogue_id', 'turn_index', 'speaker', 'is_manipulative',
    'technique', 'vulnerability', 'source_dataset', 'annotator_votes', 'label'
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
assert len(FEATURE_COLS) == N_FEATURES, f'Expected {N_FEATURES} features, got {len(FEATURE_COLS)}: {FEATURE_COLS}'

# Split by dialogue_id — no turn-level leakage
dialogue_ids = df['dialogue_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=dialogue_ids))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss2.split(df.iloc[temp_idx], groups=dialogue_ids[temp_idx]))
val_idx  = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print('Train label dist:', dict(train_df['label'].value_counts().sort_index()))
print('Val label dist:  ', dict(val_df['label'].value_counts().sort_index()))
print('Test label dist: ', dict(test_df['label'].value_counts().sort_index()))


## 3. Class Weights (Handle Imbalance)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Stage 1: Genuine (0) vs non-Genuine (1)
# Imbalance is ~1:8 (Genuine:Rest). Cap raised to 8.0 — the old 4.0 cap halved
# the balanced weight before training even started, leaving Genuine under-weighted.
s1_y = (train_df['label'] != 0).astype(int).values
s1_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=s1_y)
s1_weights = np.clip(s1_weights, None, 8.0).astype(np.float32)
print('Stage 1 weights (Genuine vs Rest):', dict(zip(['Genuine','Rest'], s1_weights.round(3))))

# Stage 2: Sarcasm (0) vs Manipulation (1) — trained only on non-Genuine rows
train_s2 = train_df[train_df['label'] != 0].copy()
train_s2['s2_label'] = train_s2['label'] - 1   # 1→0 (Sarcasm), 2→1 (Manipulation)
s2_y = train_s2['s2_label'].values
s2_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=s2_y)
s2_weights = np.clip(s2_weights, None, 4.0).astype(np.float32)
print('Stage 2 weights (Sarcasm vs Manipulation):', dict(zip(['Sarcasm','Manipulation'], s2_weights.round(3))))

# Prepare val/test stage-2 subsets
val_s2  = val_df[val_df['label']   != 0].copy(); val_s2['s2_label']  = val_s2['label']  - 1
test_s2 = test_df[test_df['label'] != 0].copy(); test_s2['s2_label'] = test_s2['label'] - 1

## 4. DataLoaders

In [ ]:
def make_loader(split_df, label_col, weights=None, balanced=False):
    X = torch.tensor(split_df[FEATURE_COLS].values, dtype=torch.float32)
    y = torch.tensor(split_df[label_col].values, dtype=torch.long)
    ds = TensorDataset(X, y)
    if balanced and weights is not None:
        sample_w = np.array([weights[l] for l in split_df[label_col].values], dtype=np.float64)
        sampler = WeightedRandomSampler(torch.tensor(sample_w), len(split_df), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

# Stage 1 loaders (full dataset, binary label: 0=Genuine, 1=Rest)
train_df['s1_label'] = (train_df['label'] != 0).astype(int)
val_df['s1_label']   = (val_df['label']   != 0).astype(int)
test_df['s1_label']  = (test_df['label']  != 0).astype(int)

s1_train_loader = make_loader(train_df, 's1_label', s1_weights, balanced=True)
s1_val_loader   = make_loader(val_df,   's1_label')
s1_test_loader  = make_loader(test_df,  's1_label')

# Stage 2 loaders (non-Genuine only, binary: 0=Sarcasm, 1=Manipulation)
s2_train_loader = make_loader(train_s2, 's2_label', s2_weights, balanced=True)
s2_val_loader   = make_loader(val_s2,   's2_label')
s2_test_loader  = make_loader(test_s2,  's2_label')

print(f'S1 train batches: {len(s1_train_loader)} | S2 train batches: {len(s2_train_loader)}')
print(f'S1 val: {len(val_df):,} rows | S2 val: {len(val_s2):,} rows (non-Genuine only)')


## 5. Model Architecture — IncongruityClassifier

In [ ]:
class StageClassifier(nn.Module):
    """
    Larger binary MLP with BatchNorm: 42 → 256 → BN → 128 → BN → 64 → 32 → n_classes
    Reused for both stages.
    BatchNorm stabilises training and reduces internal covariate shift,
    which matters here since emotion feature scales vary widely.
    """
    def __init__(self, in_dim=N_FEATURES, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),                          nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, 32),                           nn.GELU(),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.net(x)


stage1 = StageClassifier(n_classes=2).to(DEVICE)  # Genuine vs Rest
stage2 = StageClassifier(n_classes=2).to(DEVICE)  # Sarcasm vs Manipulation

total = sum(p.numel() for p in stage1.parameters())
print(f'StageClassifier params: {total:,} (per stage)')
print(f'Stage 1: Genuine(0) vs Rest(1)')
print(f'Stage 2: Sarcasm(0) vs Manipulation(1)  — applied only to Rest predictions')


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss — down-weights easy examples so the model focuses on hard ones.
    gamma=0 → vanilla CrossEntropy. gamma=2 is the standard default.
    """
    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.05):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets,
                             label_smoothing=self.label_smoothing,
                             reduction='none')
        pt = torch.exp(-ce)
        focal_weight = (1.0 - pt) ** self.gamma
        return (focal_weight * ce).mean()


criterion = FocalLoss(gamma=2.0, label_smoothing=0.05)
print('FocalLoss ready (gamma=2.0, label_smoothing=0.05)')


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

# ── train_stage: reusable MLP training loop (used for Stage 2) ───────────────
def train_stage(model, train_loader, val_loader, ckpt_path, stage_name,
                epochs=EPOCHS, patience=PATIENCE, focal_gamma=2.0):
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    criterion = FocalLoss(gamma=focal_gamma, label_smoothing=0.05)
    best_f1, patience_left, history = 0.0, patience, []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            train_loss += loss.item()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                preds.extend(model(X_batch.to(DEVICE)).argmax(1).cpu().tolist())
                trues.extend(y_batch.tolist())
        val_f1 = f1_score(trues, preds, average='macro', zero_division=0)

        improved = val_f1 > (best_f1 + 1e-4)
        if improved:
            best_f1 = val_f1; patience_left = patience
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_left -= 1

        history.append({'epoch': epoch, 'loss': train_loss/len(train_loader),
                        'val_f1': val_f1, 'best_f1': best_f1})
        if epoch % 5 == 0 or epoch == 1 or improved:
            print(f'[{stage_name}] Epoch {epoch:02d}/{epochs}  loss={train_loss/len(train_loader):.4f}  '
                  f'val_f1={val_f1:.4f}  best={best_f1:.4f}  pat={patience_left}')
        if patience_left == 0:
            print(f'[{stage_name}] Early stopping at epoch {epoch}')
            break

    print(f'[{stage_name}] Done — best val macro-F1: {best_f1:.4f}\n')
    return best_f1, history


# ── Stage 1: XGBoost — Genuine vs Rest ───────────────────────────────────────
print('Training Stage 1: Genuine vs Rest (XGBoost)')
print('=' * 60)

X_s1_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_s1_train = train_df['s1_label'].values
X_s1_val   = val_df[FEATURE_COLS].values.astype(np.float32)
y_s1_val   = val_df['s1_label'].values

sample_weights_s1 = compute_sample_weight('balanced', y_s1_train)

xgb_s1 = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=SEED,
    tree_method='hist',
    device='cpu',   # keep on CPU — data is CPU numpy, avoids device mismatch warning
)

xgb_s1.fit(
    X_s1_train, y_s1_train,
    sample_weight=sample_weights_s1,
    eval_set=[(X_s1_val, y_s1_val)],
    verbose=100,
)

val_preds_s1 = xgb_s1.predict(X_s1_val)
s1_best_f1 = f1_score(y_s1_val, val_preds_s1, average='macro', zero_division=0)
print(f'\n[S1] Done — val macro-F1: {s1_best_f1:.4f}')
xgb_s1.save_model(CKPT_S1_XGB)
print(f'Saved: {CKPT_S1_XGB}')

# ── Stage 2: MLP — Sarcasm vs Manipulation ───────────────────────────────────
print('\nTraining Stage 2: Sarcasm vs Manipulation (MLP)')
print('=' * 60)
s2_best_f1, s2_history = train_stage(stage2, s2_train_loader, s2_val_loader,
                                      CKPT_S2, 'S2', focal_gamma=2.0)

## 7. Final Test Evaluation

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Load best checkpoints
xgb_s1 = XGBClassifier()
xgb_s1.load_model(CKPT_S1_XGB)
stage2.load_state_dict(torch.load(CKPT_S2, map_location=DEVICE))
stage2.eval()


def get_X(df):
    return df[FEATURE_COLS].values.astype(np.float32)


def two_stage_predict(X_arr, s1_thresh=0.5):
    """
    Stage 1 (XGBoost): P(Genuine) >= s1_thresh → predict Genuine.
    Stage 2 (MLP):     classify the rest as Sarcasm or Manipulation.
    Accepts numpy array or torch Tensor.
    """
    if isinstance(X_arr, torch.Tensor):
        X_arr = X_arr.cpu().numpy()
    X_arr = np.asarray(X_arr, dtype=np.float32)

    # Stage 1 — XGBoost probability
    s1_probs   = xgb_s1.predict_proba(X_arr)   # (N, 2): col 0 = P(Genuine)
    is_genuine = s1_probs[:, 0] >= s1_thresh

    # Stage 2 — MLP
    with torch.no_grad():
        s2_pred = stage2(
            torch.tensor(X_arr, dtype=torch.float32).to(DEVICE)
        ).argmax(1).cpu().numpy()

    return np.where(is_genuine, 0, s2_pred + 1)


# ── Tune Stage 1 threshold on val set ────────────────────────────────────────
# Constraint: Genuine precision >= 0.40 before maximising macro-F1.
# This prevents the unconstrained optimiser from choosing a low threshold that
# tanks Genuine precision and swallows most Manipulation samples as Genuine.
val_X = get_X(val_df)
val_y = val_df['label'].values.tolist()

best_thresh, best_f1 = 0.5, 0.0
GENUINE_PREC_MIN = 0.40

print('Threshold sweep (candidates meeting Genuine precision >= 0.40):')
for thresh in np.arange(0.35, 0.90, 0.01):
    preds    = two_stage_predict(val_X, s1_thresh=thresh).tolist()
    macro_f1 = f1_score(val_y, preds, average='macro', zero_division=0)
    p, r, _, _ = precision_recall_fscore_support(val_y, preds, labels=[0,1,2], zero_division=0)
    if p[0] >= GENUINE_PREC_MIN and macro_f1 > best_f1:
        best_f1, best_thresh = macro_f1, float(thresh)
        print(f'  thresh={thresh:.2f}  macro-F1={macro_f1:.4f}  '
              f'Genuine prec={p[0]:.2f} rec={r[0]:.2f}  Manip rec={r[2]:.2f}  *best*')

if best_f1 == 0.0:
    print('Warning: precision constraint never met — falling back to unconstrained macro-F1')
    for thresh in np.arange(0.35, 0.90, 0.01):
        preds = two_stage_predict(val_X, s1_thresh=thresh).tolist()
        f1    = f1_score(val_y, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, float(thresh)

print(f'\nStage 1 threshold: {best_thresh:.2f}  (val macro-F1: {best_f1:.4f})')

# ── Apply to test set ─────────────────────────────────────────────────────────
test_X     = get_X(test_df)
test_true  = test_df['label'].values.tolist()
test_preds = two_stage_predict(test_X, s1_thresh=best_thresh).tolist()

print('\nClassification Report')
print(classification_report(test_true, test_preds, target_names=CLASS_NAMES, zero_division=0))
macro_f1 = f1_score(test_true, test_preds, average='macro', zero_division=0)
print(f'Test macro-F1: {macro_f1:.4f}  (target: > 0.60)')

## 9. Save Checkpoint + Config

In [ ]:
cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Two-Stage IncongruityClassifier\nmacro-F1 = {macro_f1:.4f}')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()


In [ ]:
import json as _json

cfg = {
    'architecture':   'Stage1=XGBoost 46-dim | Stage2=MLP 46→256→BN→128→BN→64→32→2',
    'n_features':     N_FEATURES,
    'n_classes':      N_CLASSES,
    'class_names':    CLASS_NAMES,
    'stage1_ckpt':    CKPT_S1_XGB,
    'stage2_ckpt':    CKPT_S2,
    's1_threshold':   best_thresh,
    's1_best_val_f1': s1_best_f1,
    's2_best_val_f1': s2_best_f1,
    'test_macro_f1':  macro_f1,
    'feature_cols':   FEATURE_COLS,
    'note':           'Stage1=XGBoost Genuine vs Rest (threshold-tuned); Stage2=MLP Sarcasm vs Manipulation',
}
with open(os.path.join(OUT_DIR, 'model2_config.json'), 'w') as f:
    _json.dump(cfg, f, indent=2)

# XGBoost S1 has no per-epoch history — save S2 MLP history only
pd.DataFrame(s2_history).to_csv(os.path.join(OUT_DIR, 'model2_history.csv'), index=False)

print(f'Saved: {CKPT_S1_XGB}')
print(f'Saved: {CKPT_S2}')
print(f'Stage 1 threshold: {best_thresh:.2f}  (stored in model2_config.json)')
print('Share stage1_genuine_vs_rest.json + stage2_sarcasm_vs_manip.pt as Kaggle dataset: incongruity-classifier')

In [ ]:
# ── 10. Qualitative Test: Multi-turn Conversation ─────────────────────────────
import json, re
from transformers import AutoTokenizer, AutoModel

_m1_dir = None
for _base in ['/kaggle/input/plutchik-model-v2',
              '/kaggle/input/datasets/bobhendriks/plutchik-model-v2']:
    if os.path.isdir(_base):
        for _r, _, _fs in os.walk(_base):
            if 'model_config.json' in _fs:
                _m1_dir = _r; break
    if _m1_dir: break
if _m1_dir is None:
    raise FileNotFoundError('Add plutchik-model-v2 as a Kaggle input to run this cell.')

class _EAB(nn.Module):
    def __init__(self, H, nh=4, out=128):
        super().__init__()
        hd = H // nh; self.nh = nh; self.sc = hd**-0.5
        self.query    = nn.Parameter(torch.randn(1,1,H)*0.02)
        self.k_proj   = nn.Linear(H,H,bias=False)
        self.v_proj   = nn.Linear(H,H,bias=False)
        self.out_proj = nn.Linear(H,out)
        self.norm     = nn.LayerNorm(out)
        self.drop     = nn.Dropout(0.1)
    def forward(self, h, mask):
        B,L,H = h.shape; nh,hd = self.nh, H//self.nh
        q = self.query.expand(B,-1,-1)
        k,v = self.k_proj(h), self.v_proj(h)
        q=q.view(B,1,nh,hd).transpose(1,2); k=k.view(B,L,nh,hd).transpose(1,2); v=v.view(B,L,nh,hd).transpose(1,2)
        a = (q@k.transpose(-2,-1))*self.sc
        if mask is not None: a = a.masked_fill((mask==0).unsqueeze(1).unsqueeze(2),float('-inf'))
        return self.norm(self.out_proj((self.drop(torch.softmax(a,-1))@v).squeeze(2).reshape(B,H)))

class _PM(nn.Module):
    def __init__(self, base, ne=8, out=128, dr=0.1):
        super().__init__()
        self.encoder        = AutoModel.from_pretrained(base); H = self.encoder.config.hidden_size
        self.emotion_blocks = nn.ModuleList([_EAB(H,4,out) for _ in range(ne)])
        self.emotion_heads  = nn.ModuleList([nn.Sequential(nn.Linear(out,32),nn.GELU(),nn.Dropout(dr),nn.Linear(32,1)) for _ in range(ne)])
        self.temperature    = nn.Parameter(torch.full((ne,),1.0)); self.dr = nn.Dropout(dr)
        self.conf_head      = nn.Sequential(nn.Linear(H,128),nn.GELU(),nn.Dropout(dr),nn.Linear(128,1),nn.Sigmoid())
        self.cls_head       = nn.Sequential(nn.Linear(H,128),nn.GELU(),nn.Dropout(dr),nn.Linear(128,ne))
    def forward(self, ids, mask, tti=None):
        h = self.dr(self.encoder(input_ids=ids,attention_mask=mask,token_type_ids=tti).last_hidden_state)
        lg = torch.cat([head(blk(h,mask)) for blk,head in zip(self.emotion_blocks,self.emotion_heads)], dim=-1)
        return torch.sigmoid(lg*torch.clamp(self.temperature,0.5,5.0)), self.conf_head(h[:,0]).squeeze(-1), self.cls_head(h[:,0])

with open(os.path.join(_m1_dir,'model_config.json')) as f: _cfg1 = json.load(f)
_tok = AutoTokenizer.from_pretrained(_m1_dir)
_m1  = _PM(_cfg1['base_model']).to(DEVICE).float()
_m1.load_state_dict(torch.load(os.path.join(_m1_dir,'best_model.pt'), map_location=DEVICE))
_m1.eval()

# Reload Stage 1 XGBoost and Stage 2 MLP (safe to re-run cell independently)
_xgb_s1 = XGBClassifier(); _xgb_s1.load_model(CKPT_S1_XGB)
stage2.load_state_dict(torch.load(CKPT_S2, map_location=DEVICE)); stage2.eval()
print(f'Models loaded  |  S1 threshold = {best_thresh:.2f}')

# ── Lexical patterns (must match NB6) ─────────────────────────────────────────
_NEG  = re.compile(r"\b(not|never|no\b|don't|doesn't|didn't|can't|won't|isn't|aren't|wasn't|weren't|nothing|nobody|nowhere|neither|nor)\b", re.IGNORECASE)
_PRES = re.compile(r"\b(everyone|everybody|always|supposed|should|must|expected|assumed|awkward|obviously|clearly|of\s+course|you\s+always|you\s+never|disappointed|I\s+assumed|I\s+expected)\b", re.IGNORECASE)
_HEDG = re.compile(r"\b(maybe|perhaps|possibly|probably|might|could\s+be|seems|apparently|I\s+think|I\s+feel|I\s+guess|sort\s+of|kind\s+of|I\s+suppose|I\s+believe)\b", re.IGNORECASE)

def _lex(text):
    text = str(text); words = text.split(); n = max(len(words), 1)
    return np.array([
        min(len(_NEG.findall(text))  / n, 1.0),
        min(len(_PRES.findall(text)) / n, 1.0),
        min(len(_HEDG.findall(text)) / n, 1.0),
        float(bool(re.search(r'[!?]', text))),
    ], dtype=np.float32)

_OPAIRS = [(0,4),(1,5),(2,6),(3,7)]

def _vlit(text):
    enc = _tok([text], max_length=128, padding='max_length', truncation=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        s,_,_ = _m1(enc['input_ids'], enc['attention_mask'], enc.get('token_type_ids'))
    return s[0].float().cpu().numpy()

class _Trk:
    def __init__(self):
        self.ctx = np.zeros(8); self.spk = {}; self.hist = []
    def update(self, v, s):
        self.hist.append(self.ctx.copy())
        if len(self.hist)>3: self.hist.pop(0)
        self.ctx = 0.3*v + 0.7*self.ctx
        self.spk[s] = 0.3*v + 0.7*self.spk.get(s, v.copy())
    def deltas(self, v):
        h = self.hist
        return (v-h[-1] if len(h)>=1 else np.zeros(8),
                v-h[-2] if len(h)>=2 else np.zeros(8),
                v-h[-3] if len(h)>=3 else np.zeros(8))
    def drift(self,v):   return float(np.linalg.norm(v-self.ctx))
    def actor(self,v,s): return float(np.linalg.norm(self.spk.get(s,self.ctx)-self.ctx))

def _feats(v, trk, spk, text=''):
    """Build 46-dim feature vector matching NB6 layout."""
    d1,d2,d3 = trk.deltas(v)
    pt  = [float(v[i])*float(v[j]) for i,j in _OPAIRS]
    vn  = v+1e-8; vn=vn/vn.sum()
    return np.concatenate([v, d1, d2, d3, pt, [
        trk.drift(v), trk.actor(v,spk),
        float(np.max(v)-np.mean(v)),
        sum(float(v[i])*float(v[j]) for i,j in _OPAIRS),
        float(1-np.max(v)) if np.max(v)<0.3 else 0.0,
        float(-np.sum(vn*np.log(vn))/np.log(8)),
    ], _lex(text)]).astype(np.float32)

conversation = [
    ("Alice", "Genuine",      "Hey, did you get a chance to look at the project brief I sent over?"),
    ("Bob",   "Genuine",      "Yeah I went through it this morning, looks pretty straightforward to me."),
    ("Alice", "Genuine",      "Great, let me know if you need any clarification on the timeline."),
    ("Bob",   "Sarcasm",      "Oh absolutely, because the timeline makes total sense when half the team is on leave."),
    ("Alice", "Genuine",      "Fair point, we can push the deadline if needed."),
    ("Bob",   "Sarcasm",      "Sure, because pushing deadlines has worked out so well for us every single time."),
    ("Alice", "Manipulation", "I've already told the director you'd have this wrapped up by Friday — it'd be really awkward if that changed now."),
    ("Bob",   "Genuine",      "I'll do my best, but I can't promise Friday if the scope keeps growing."),
    ("Alice", "Manipulation", "Everyone else manages to deliver on time. I just assumed you'd be the same."),
    ("Bob",   "Sarcasm",      "Right, everyone delivers on time. Totally what happens in this team."),
]

trk = _Trk()
print(f'\n{"Spk":<6} {"Expected":<13} {"Predicted":<13}  Text')
print('─' * 85)
n_correct = 0
for speaker, expected, text in conversation:
    v    = _vlit(text)
    feat = _feats(v, trk, speaker, text=text)
    trk.update(v, speaker)

    x = feat.reshape(1, -1)
    s1_genuine_prob = _xgb_s1.predict_proba(x)[0, 0]
    if s1_genuine_prob >= best_thresh:
        pred_idx = 0
    else:
        with torch.no_grad():
            pred_idx = int(stage2(torch.tensor(x, dtype=torch.float32).to(DEVICE)).argmax(1).item()) + 1
    pred = CLASS_NAMES[pred_idx]
    ok   = pred == expected
    n_correct += ok
    print(f'{"✓" if ok else "✗"} {speaker:<5} {expected:<13} {pred:<13}  {text[:55]}')

print(f'\n{n_correct}/{len(conversation)} correct')